In [2]:
import pandas as pd

In [40]:
yelp = pd.read_excel("Yelp dataset.xlsx")
yelp.head()

,Review_id,Product_id,Review_Date,Review_Text
0,923,0,2014-12-08,The food at snack is a selection of popular Gr...
1,924,0,2013-05-16,This little place in Soho is wonderful. I had ...
2,925,0,2013-07-01,ordered lunch for 15 from Snack last Friday. Â...
3,926,0,2011-07-28,This is a beautiful quaint little restaurant o...
4,927,0,2010-11-01,Snack is great place for a Â casual sit down l...


In [41]:
yelp_meta = pd.read_excel("Yelp Metadata.xlsx")
yelp_meta.head()

,Review_id,Product_id,Rating,Label,Review_Date
0,923,0,3,-1,2014-12-08
1,924,0,3,-1,2013-05-16
2,925,0,4,-1,2013-07-01
3,926,0,4,-1,2011-07-28
4,927,0,4,-1,2010-11-01


In [42]:
# yelp, yelp_meta 병합 
merged = pd.DataFrame()

merged["user_id"] = yelp["Review_id"]
merged["product_id"] = yelp["Product_id"]
merged["rating"] = yelp_meta["Rating"]
merged["date"] = yelp["Review_Date"]
merged["review_text"] = yelp["Review_Text"]
merged["label"] = yelp_meta["Label"]

merged.head()

,user_id,product_id,rating,date,review_text,label
0,923,0,3,2014-12-08,The food at snack is a selection of popular Gr...,-1
1,924,0,3,2013-05-16,This little place in Soho is wonderful. I had ...,-1
2,925,0,4,2013-07-01,ordered lunch for 15 from Snack last Friday. Â...,-1
3,926,0,4,2011-07-28,This is a beautiful quaint little restaurant o...,-1
4,927,0,4,2010-11-01,Snack is great place for a Â casual sit down l...,-1


In [ ]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 359052 entries, 0 to 359051
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   user_id      359052 non-null  int64         
 1   product_id   359052 non-null  int64         
 2   rating       359052 non-null  int64         
 3   date         359052 non-null  datetime64[ns]
 4   review_text  359052 non-null  object        
 5   label        359052 non-null  int64         
dtypes: datetime64[ns](1), int64(4), object(1)
memory usage: 16.4+ MB


In [ ]:
merged["label"].value_counts()

label
 1    322167
-1     36885
Name: count, dtype: int64

In [ ]:
# 필요한 컬럼만 사용
review_text = "review_text"
label = "label"

# 결측치 제거
df = merged.dropna()
df[review_text] = df[review_text].astype(str)

# 라벨을 0/1로 매핑
df[label] = df[label].map({1: 0, -1: 1}).astype(int)

print("데이터셋 크기:", len(df))
print("진짜 리뷰 수:", df[df[label] == 0].shape[0])
print("가짜 리뷰 수:", df[df[label] == 1].shape[0])

데이터셋 크기: 359052
진짜 리뷰 수: 322167
가짜 리뷰 수: 36885


In [3]:
import re
import pandas as pd
import nltk
import spacy
from textstat import textstat

nltk.download('punkt')

nlp = spacy.load("en_core_web_sm")

/opt/anaconda3/envs/review-detector/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
[nltk_data] Downloading package punkt to /Users/jooyoung/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [58]:
def clean_review(text):
    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(r"<.*?>", " ", text)          # HTML 제거
    text = re.sub(r"http\S+|www\S+", " ", text) # URL 제거
    text = re.sub(r"\S+@\S+", " ", text)        # 이메일 제거
    text = re.sub(r"[\n\r\t]", " ", text)       # 줄바꿈 제거
    text = re.sub(r"\s+", " ", text).strip()    # 공백 정리

    return text

df["clean_review"] = df["review_text"].apply(clean_review)

# Basic linguisitic features
## syllable (총 음절 수)

In [59]:
# 리뷰 텍스트의 모든 총 음절 개수
def get_syllable(text):
    words = nltk.word_tokenize(str(text))
    words = [w for w in words if w.isalpha()]
    return sum(textstat.syllable_count(w) for w in words)

df["syllable"] = df["clean_review"].apply(get_syllable)

## lexicon (총 어휘 수)

In [60]:
# 리뷰 텍스트 내에 사용된 모든 단어의 총 개수
def get_lexicon(text):
    words = nltk.word_tokenize(str(text))
    words = [w for w in words if w.isalpha()]
    return len(words)

df["lexicon"] = df["clean_review"].apply(get_lexicon)

## sentencet (총 문장 수)

In [62]:
# 리뷰 텍스트 내 문장의 총 개수
def get_sentencet(text):
    return len(nltk.sent_tokenize(str(text)))

df["sentencet"] = df["clean_review"].apply(get_sentencet)

## char (총 문자 수)

In [63]:
# 리뷰 텍스트 내 공백을 포함한 모든 문자의 총 개수
def get_char(text):
    return len(str(text))

df["char"] = df["clean_review"].apply(get_char)

## letter (총 글자 수)

In [64]:
# 리뷰 텍스트 내 알파벳 글자의 총 개수
def get_letter(text):
    return sum(c.isalpha() for c in str(text))

df["letter"] = df["clean_review"].apply(get_letter)

## polysyllab (총 다음절 단어 수)

In [65]:
# 리뷰 텍스트 내 세 개 이상의 음절을 가진 단어의 총 개수
def get_polysyllab(text):
    words = nltk.word_tokenize(str(text))
    words = [w for w in words if w.isalpha()]
    
    count = 0
    for w in words:
        if textstat.syllable_count(w) >= 3:
            count += 1
            
    return count

df["polysyllab"] = df["clean_review"].apply(get_polysyllab)

## monosyllab (총 단음절 단어 수)

In [66]:
# 리뷰 텍스트 내 하나의 음절을 가진 단어의 총 개수
def get_monosyllab(text):
    words = nltk.word_tokenize(str(text))
    words = [w for w in words if w.isalpha()]
    
    count = 0
    for w in words:
        if textstat.syllable_count(w) == 1:
            count += 1
            
    return count

df["monosyllab"] = df["clean_review"].apply(get_monosyllab)

## nouns (총 명사 수)

In [67]:
# 리뷰 텍스트 내 명사의 총 개수
def get_nouns(text):
    doc = nlp(str(text))
    return sum(1 for token in doc if token.pos_ == "NOUN")

df["nouns"] = df["clean_review"].apply(get_nouns)

## adj (총 형용사 수)

In [68]:
# 리뷰 텍스트 내 형용사의 총 개수
def get_adj(text):
    doc = nlp(str(text))
    return sum(1 for token in doc if token.pos_ == "ADJ")

df["adj"] = df["clean_review"].apply(get_adj)

## verbs (총 동사 수)

In [69]:
# 리뷰 텍스트 내 동사의 총 개수
def get_verbs(text):
    doc = nlp(str(text))
    return sum(1 for token in doc if token.pos_ == "VERB")

df["verbs"] = df["clean_review"].apply(get_verbs)

## pronoun (총 대명사 수)

In [70]:
# 리뷰 텍스트 내 대명사의 총 개수
def get_pronoun(text):
    doc = nlp(str(text))
    return sum(1 for token in doc if token.pos_ == "PRON")

df["pronoun"] = df["clean_review"].apply(get_pronoun)

## adverb (총 부사 수)

In [71]:
# 리뷰 텍스트 내 부사의 총 개수
def get_adverb(text):
    doc = nlp(str(text))
    return sum(1 for token in doc if token.pos_ == "ADV")

df["adverb"] = df["clean_review"].apply(get_adverb)

## article (총 관사 수)

In [72]:
# 리뷰 텍스트 내 관사(a, an, the 등)의 총 개수
def get_article(text):
    doc = nlp(str(text))
    return sum(1 for token in doc if token.lower_ in ["a", "an", "the"])

df["article"] = df["clean_review"].apply(get_article)

# Readability features
## SMOG Index (SMOG 지수)

In [73]:
# 텍스트의 가독성 수준을 측정하여 해당 텍스트를 이해하는 데 필요한 교육 연수 추정 지표
def get_smog(text):
    text = str(text)
    if len(text.split()) < 10:
        return 0
    return textstat.smog_index(text)

df["smog"] = df["clean_review"].apply(get_smog)

## Flesch_Reading_Ease (Flesch 읽기 용이성 점수)

In [74]:
# 텍스트가 얼마나 읽기 쉬운지를 0점에서 100점까지의 척도로 나타내는 지표
def get_flesch_reading_ease(text):
    return textstat.flesch_reading_ease(str(text))

df["flesch_reading_ease"] = df["clean_review"].apply(get_flesch_reading_ease)

## Flesch_Kincaid_Grade (Flesch Kincaid 학년 점수)

In [75]:
# Flesch 읽기 용이성 점수를 미국 학년 수준으로 변환한 것
def get_flesch_kincaid_grade(text):
    return textstat.flesch_kincaid_grade(str(text))

df["flesch_kincaid_grade"] = df["clean_review"].apply(get_flesch_kincaid_grade)

## Fog_Scale (Fog 지수)

In [76]:
# 텍스트를 처음 읽을 때 이해하는 데 필요한 정규 교육 연수를 추정하는 지수
def get_fog_scale(text):
    return textstat.gunning_fog(str(text))

df["fog_scale"] = df["clean_review"].apply(get_fog_scale)

## Dale_Chall_Readability (Dale-Chall 가독성 점수)

In [77]:
# 단어의 친숙도를 기반으로 가독성을 측정한 지표
def get_dale_chall_readability(text):
    return textstat.dale_chall_readability_score(str(text))

df["dale_chall_readability"] = df["clean_review"].apply(get_dale_chall_readability)

## Reading_Time (읽기 시간)

In [78]:
# 평균적인 독자가 텍스트를 읽는 데 걸리는 시간을 추정한 것
def get_reading_time(text):
    return textstat.reading_time(str(text))

df["reading_time"] = df["clean_review"].apply(get_reading_time)

# Emotional lexical features

In [4]:
from textblob import TextBlob
from nrclex import NRCLex
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from vaderSentiment.vaderSentiment import NEGATE

## sentiment (감정 점수)

In [93]:
# 리뷰 텍스트의 전반적인 감정적 극성을 나타내는 점수
def get_sentiment(text):
    return TextBlob(str(text)).sentiment.polarity

df["sentiment"] = df["clean_review"].apply(get_sentiment)

## subjectivity (주관성 점수)

In [94]:
# 리뷰 텍스트가 얼마나 주관적인 의견이나 감정을 담고 있는지를 나타내는 점수
def get_subjectivity(text):
    return TextBlob(str(text)).sentiment.subjectivity

df["subjectivity"] = df["clean_review"].apply(get_subjectivity)

## anger (총 분노 단어 수)

In [6]:
# 리뷰 텍스트에 포함된 '분노'와 관련된 어휘의 총 개수
def get_anger(text):
    emotion = NRCLex(str(text))
    return emotion.raw_emotion_scores.get("anger", 0)

df["anger"] = df["clean_review"].apply(get_anger)

## sadness (총 슬픔 단어 수)

In [7]:
# 리뷰 텍스트에 포함된 '슬픔'과 관련된 어휘의 총 개수
def get_sadness(text):
    emotion = NRCLex(str(text))
    return emotion.raw_emotion_scores.get("sadness", 0)

df["sadness"] = df["clean_review"].apply(get_sadness)

## posemo (총 긍정 감정 단어 수)

In [8]:
# 리뷰 텍스트에 포함된 '긍정적'인 감정을 나타내는 어휘의 총 개수
def get_posemo(text):
    emotion = NRCLex(str(text))
    return emotion.raw_emotion_scores.get("positive", 0)

df["posemo"] = df["clean_review"].apply(get_posemo)

## negate (총 부정 단어 수)

In [9]:
# 리뷰 텍스트에 포함된 '부정' 또는 '부인'을 나타내는 어휘의 총 개수
def get_negate(text):
    words = nltk.word_tokenize(str(text).lower())
    return sum(1 for w in words if w in NEGATE)

df["negate"] = df["clean_review"].apply(get_negate)

## anx (총 불안 단어 수)

In [10]:
# 리뷰 텍스트에 포함된 '불안'과 관련된 어휘의 총 개수
def get_anx(text):
    emotion = NRCLex(str(text))
    return emotion.raw_emotion_scores.get("fear", 0)

df["anx"] = df["clean_review"].apply(get_anx)

In [49]:
df.head()

,review_text,user_id,product_id,rating,date,label,clean_review,syllable,lexicon,sentencet,...,fog_scale,dale_chall_readability,reading_time,sentiment,subjectivity,anger,sadness,posemo,negate,anx
0,The food at snack is a selection of popular Gr...,923,0,3,2014-12-08,1,The food at snack is a selection of popular Gr...,54.0,39.0,4.0,...,8.000000,9.264250,2.58544,0.195833,0.395833,0.0,0.0,3.0,0.0,0.0
1,This little place in Soho is wonderful. I had ...,924,0,3,2013-05-16,1,This little place in Soho is wonderful. I had ...,63.0,51.0,4.0,...,5.200000,6.710531,3.20242,0.025000,0.650000,0.0,0.0,4.0,0.0,0.0
2,ordered lunch for 15 from Snack last Friday. Â...,925,0,4,2013-07-01,1,ordered lunch for 15 from Snack last Friday. Â...,45.0,33.0,3.0,...,5.709804,8.378339,2.15943,0.220000,0.328718,0.0,1.0,1.0,1.0,1.0
3,This is a beautiful quaint little restaurant o...,926,0,4,2011-07-28,1,This is a beautiful quaint little restaurant o...,126.0,90.0,7.0,...,7.809524,7.607659,5.90538,0.555134,0.776786,0.0,0.0,10.0,0.0,0.0
4,Snack is great place for a Â casual sit down l...,927,0,4,2010-11-01,1,Snack is great place for a Â casual sit down l...,149.0,107.0,6.0,...,8.714486,8.579433,7.27155,0.138715,0.538294,0.0,0.0,10.0,1.0,0.0


In [50]:
df.to_excel("Yelp_feat.xlsx", index=False)

In [10]:
df = pd.read_excel("Yelp_feat.xlsx")

# Behavioral features

In [11]:
from scipy.stats import entropy

## review count (리뷰 수)

In [12]:
# 한 사용자가 플랫폼에 작성한 총 리뷰 개수
def get_review_count(user_id):
    return df[df['user_id'] == user_id].shape[0]

df["review_count"] = df['user_id'].apply(get_review_count)

## user tenure (사용자 활동 기간)

In [13]:
# 사용자의 활동 기간: 첫 리뷰 작성일부터 마지막 리뷰 작성일까지의 일수
def get_user_tenure(user_id):
    user_dates = df[df['user_id'] == user_id]['date']
    return (user_dates.max() - user_dates.min()).days

df["user_tenure"] = df['user_id'].apply(get_user_tenure)

## review gap avg (평균 리뷰 간격)

In [14]:
# 사용자별 리뷰 작성 간격 계산
df = df.sort_values(['user_id', 'date'])

df["review_gap"] = df.groupby('user_id')['date'].diff().dt.days

In [15]:
# 사용자가 리뷰를 작성한 평균 시간 간격
def get_review_gap_avg(user_id):
    gaps = df[df['user_id'] == user_id]["review_gap"]
    return gaps.mean()

df["review_gap_avg"] = df['user_id'].apply(get_review_gap_avg)
df["review_gap_avg"] = df["review_gap_avg"].fillna(0)

## review gap std (리뷰 간격 표준편차)

In [16]:
# 사용자의 리뷰 작성 간격 표준편차
def get_review_gap_std(user_id):
    gaps = df[df['user_id'] == user_id]["review_gap"]
    return gaps.std()

df["review_gap_std"] = df['user_id'].apply(get_review_gap_std)
df["review_gap_std"] = df["review_gap_std"].fillna(0)

## rating entropy (평점 엔트로피)

In [17]:
# 사용자가 부여한 평점의 다양성 또는 예측 불가능성
def get_rating_entropy(ratings):
    counts = ratings.value_counts(normalize=True)
    return entropy(counts)

df["rating_entropy"] = df.groupby('user_id')['rating'].transform(get_rating_entropy)

## rating devitation avg (평균 평점 편차)

In [18]:
# 사용자별 평균 평점
df["user_rating_avg"] = df.groupby('user_id')['rating'].transform("mean")

# 개별 리뷰 평점이 사용자 평균 평점에서 벗어난 정도
df["rating_deviation"] = (df['rating'] - df["user_rating_avg"]).abs()

In [19]:
# 사용자의 평균 평점 편차
def get_rating_deviation_avg(user_id):
    deviations = df[df['user_id'] == user_id]["rating_deviation"]
    return deviations.mean()

df["rating_deviation_avg"] = df['user_id'].apply(get_rating_deviation_avg)

## rating devitation std (평점 편차 표준편차)

In [20]:
# 사용자의 평점 편차 표준편차
def get_rating_deviation_std(user_id):
    deviations = df[df['user_id'] == user_id]["rating_deviation"]
    return deviations.std()

df["rating_deviation_std"] = df['user_id'].apply(get_rating_deviation_std)
df["rating_deviation_std"] = df["rating_deviation_std"].fillna(0)

## rating score avg (평균 평점)

In [21]:
# 사용자가 부여한 모든 평점의 평균값
def get_rating_scores_avg(user_id):
    ratings = df[df['user_id'] == user_id]['rating']
    return ratings.mean()

df["rating_scores_avg"] = df['user_id'].apply(get_rating_scores_avg)

## rating score std (평점 표준편차)

In [22]:
# 사용자가 부여한 평점 점수들의 표준편차
def get_rating_scores_std(user_id):
    ratings = df[df['user_id'] == user_id]['rating']
    return ratings.std()

df["rating_scores_std"] = df['user_id'].apply(get_rating_scores_std)
df["rating_scores_std"] = df["rating_scores_std"].fillna(0)

In [25]:
feature_cols = [
    "syllable", "lexicon", "sentencet", "char", "letter", "polysyllab",
    "monosyllab", "nouns", "adj", "verbs", "pronoun", "adverb", "article",
    "smog", "flesch_reading_ease", "flesch_kincaid_grade", "fog_scale",
    "dale_chall_readability", "reading_time", "sentiment", "subjectivity",
    "anger", "sadness", "posemo", "negate", "anx",
    "review_count", "user_tenure", "review_gap_avg", "review_gap_std",
    "rating_entropy", "rating_deviation_avg", "rating_deviation_std",
    "time_of_review_avg", "time_of_review_std", "rating_scores_avg",
    "rating_scores_std"
]

existing_cols = [col for col in feature_cols if col in df.columns]
df[existing_cols] = df[existing_cols].astype(float)

In [26]:
df.head()

,review_text,user_id,product_id,rating,date,label,clean_review,syllable,lexicon,sentencet,...,review_gap,review_gap_avg,review_gap_std,rating_entropy,user_rating_avg,rating_deviation,rating_deviation_avg,rating_deviation_std,rating_scores_avg,rating_scores_std
289698,"The falafel were superb, stuffed grape leaved ...",923,759,5,2013-11-04,1,"The falafel were superb, stuffed grape leaved ...",86.0,60.0,3.0,...,NaN,10.5,11.820161,0.965329,4.435897,0.564103,0.665352,0.521252,4.435897,0.852083
175032,The food is simply excellent. Everything is as...,923,131,5,2013-11-11,1,The food is simply excellent. Everything is as...,41.0,28.0,4.0,...,7.0,10.5,11.820161,0.965329,4.435897,0.564103,0.665352,0.521252,4.435897,0.852083
237712,This place is amazing.We really love good lati...,923,622,5,2013-11-19,1,This place is amazing.We really love good lati...,112.0,81.0,12.0,...,8.0,10.5,11.820161,0.965329,4.435897,0.564103,0.665352,0.521252,4.435897,0.852083
246545,I had Nasi Lemak and Nyonya Seafood Fried Rice...,923,906,5,2013-11-19,1,I had Nasi Lemak and Nyonya Seafood Fried Rice...,134.0,90.0,5.0,...,0.0,10.5,11.820161,0.965329,4.435897,0.564103,0.665352,0.521252,4.435897,0.852083
73868,Tough place to find unless you know the exact ...,923,808,5,2013-11-27,1,Tough place to find unless you know the exact ...,87.0,59.0,6.0,...,8.0,10.5,11.820161,0.965329,4.435897,0.564103,0.665352,0.521252,4.435897,0.852083


In [27]:
drop_cols = ["review_text", "user_id", "product_id", "date", "rating"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [31]:
# review_gap, review_deviation 컬럼 삭제
df = df.drop(columns=["review_gap", "rating_deviation"])

In [37]:
basic_linguistic_list = [
    "syllable",
    "lexicon",
    "sentencet",
    "char",
    "letter",
    "polysyllab",
    "monosyllab",
    "nouns",
    "adj",
    "verbs",
    "pronoun",
    "adverb",
    "article"
]

df["basic_linguistic_list"] = df[basic_linguistic_list].values.tolist()

In [38]:
readability_list = [
    "smog",
    "flesch_reading_ease",
    "flesch_kincaid_grade",
    "fog_scale",
    "dale_chall_readability",
    "reading_time"
]

df["readability_list"] = df[readability_list].values.tolist()

In [39]:
sentiment_list = [
    "sentiment",
    "subjectivity",
    "anger",
    "sadness",
    "posemo",
    "negate",
    "anx"
]

df["sentiment_list"] = df[sentiment_list].values.tolist()

In [40]:
behavioral_list = [
    "review_count",
    "user_tenure",
    "review_gap_avg",
    "review_gap_std",
    "rating_entropy",
    "rating_deviation_avg",
    "rating_deviation_std",
    "rating_scores_avg",
    "rating_scores_std"
]

df["behavioral_list"] = df[behavioral_list].values.tolist()

In [48]:
df.to_parquet("final_data_categorized.parquet", index=False)

In [54]:
df = pd.read_parquet("final_data_categorized.parquet")

In [55]:
df.head()

,review_text,fake,basic_linguistic_list,readability_list,sentiment_list,behavioral_list
0,"The falafel were superb, stuffed grape leaved ...",1,"[86.0, 60.0, 3.0, 347.0, 278.0, 6.0, 40.0, 9.0...","[11.20814326018867, 65.27500000000003, 9.12333...","[0.3917948717948718, 0.517051282051282, 2.0, 0...","[39.0, 399.0, 10.5, 11.820161429363655, 0.9653..."
1,The food is simply excellent. Everything is as...,1,"[41.0, 28.0, 4.0, 156.0, 125.0, 4.0, 19.0, 3.0...","[8.841846274778883, 75.8514285714286, 4.418571...","[0.792, 0.808, 0.0, 0.0, 4.0, 0.0, 0.0]","[39.0, 399.0, 10.5, 11.820161429363655, 0.9653..."
2,This place is amazing.We really love good lati...,1,"[112.0, 81.0, 12.0, 467.0, 360.0, 5.0, 60.0, 1...","[8.841846274778883, 73.20353658536588, 5.95905...","[0.1654761904761905, 0.5793650793650793, 2.0, ...","[39.0, 399.0, 10.5, 11.820161429363655, 0.9653..."
3,I had Nasi Lemak and Nyonya Seafood Fried Rice...,1,"[134.0, 90.0, 5.0, 520.0, 416.0, 12.0, 62.0, 1...","[11.97924847333083, 62.605, 8.998888888888889,...","[0.3452777777777778, 0.55125, 0.0, 1.0, 6.0, 1...","[39.0, 399.0, 10.5, 11.820161429363655, 0.9653..."
4,Tough place to find unless you know the exact ...,1,"[87.0, 59.0, 6.0, 334.0, 269.0, 6.0, 40.0, 12....","[8.841846274778883, 72.1050141242938, 5.645000...","[0.3525396825396825, 0.6380952380952382, 0.0, ...","[39.0, 399.0, 10.5, 11.820161429363655, 0.9653..."


In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 359046 entries, 0 to 359045
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   review_text            359046 non-null  object
 1   fake                   359046 non-null  int64 
 2   basic_linguistic_list  359046 non-null  object
 3   readability_list       359046 non-null  object
 4   sentiment_list         359046 non-null  object
 5   behavioral_list        359046 non-null  object
dtypes: int64(1), object(5)
memory usage: 16.4+ MB
